In [55]:
import pandas as pd
import numpy as np

df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


In [56]:
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


In [57]:
# Cleaning Headers

print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(' ', '_')

print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [58]:
# Type Conversion & Currency Cleaning

dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("Fix Applied")
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))


   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
Fix Applied
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [59]:
# Categorical Typos (Fuzzy Logic)

print(df['channel'].unique())

cleanup_map = {
    'Tik_Tok': 'TikTok',
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'E-mail': 'Email',
    'Gogle':'Google Ads',
    'N/A': np.nan
}

df['channel'] = df['channel'].replace(cleanup_map)

print("Fix Applied")
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
Fix Applied
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


In [60]:
# Handling Mixed Booleans

print(df['active'].unique())

bool_map = {
    'Yes': True,
    'Y': True,
    '1':True,
    1: True,
    'No': False,
    'N': False,
    '0': False,
    'N/A': np.nan
}

df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

print("Fix Applied")
print(df['active'].unique())

['Y' '0' 'No' 'True' 'Yes' '1' 'False']
Fix Applied
[ True False]


/tmp/ipykernel_1328/3172210651.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)


In [61]:
# Date Parsing

print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("Fix Applied")
print(df['start_date'].dtype)

object
Fix Applied
datetime64[ns]


/tmp/ipykernel_1328/2844919412.py:6: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


In [62]:
# Remove duplicate columns

df = df.loc[:, ~df.columns.duplicated()]

In [63]:
# Logical Integrrity (Click vs Impressions)

impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask,['campaign_id', 'impressions', 'clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [64]:
# Logical Integrity (Time Travel)

time_travel_mask = df['start_date'] < df['end_date']
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)
print("Fix Applied")
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))


  campaign_id start_date   end_date
0   CMP-00001 2023-11-24 2023-12-13
1   CMP-00002 2023-05-06 2023-05-12
2   CMP-00003 2023-12-13 2023-12-20
Fix Applied
  campaign_id start_date   end_date
0   CMP-00001 2023-11-24 2023-12-24
1   CMP-00002 2023-05-06 2023-06-05
2   CMP-00003 2023-12-13 2024-01-12


In [65]:
# Handling Outliers (Winsorizing)

Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

print("Fix Applied")
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))


     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
Fix Applied
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


In [71]:
# String Parsing (Feature Extraction)

print(df['campaign_name'].head(3))

df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print("Fix Applied")

print(df[['campaign_name', 'season']].head(3))


0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
Fix Applied
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [66]:
9